# Intro to Data Science & Basic Machine Learning

Welcome to the CS+Social Good workshop!

Today we'll use a fictional bike rental dataset to explore the basic data science pipeline and build our first machine learning models.

**Our questions:**
1. Can we predict whether a day will be busy?
2. Can we predict how many bikes will be rented?

Our dataset contains 180 simulated days of weather and rental information.

In [ ]:
%pip install pandas matplotlib scikit-learn

## 1. Load Our Dataset

We'll use **pandas**, a Python library for working with tables of data.

Each row represents one day, and each column represents a different piece of information.

In [ ]:
import pandas as pd

df = pd.read_csv("bike_rentals.csv")
df.head()

In [ ]:
print("Rows and columns:", df.shape)
print("Column names:", df.columns.tolist())

## 2. Data Cleaning

Real datasets often contain missing or incorrect values.

Three common approaches:
- **Mean imputation:** Replace a missing number with the average.
- **Mode imputation:** Replace a missing value with the most common value.
- **Dropping:** Remove rows containing missing values.

Let's see what's missing!

In [ ]:
df.isnull().sum()

In [ ]:
print("Average temperature:", df["temperature_f"].mean())
print("Most common rain type:", df["rain"].mode()[0])
print("Rows without missing values:", len(df.dropna()))

**Discussion:** Should we drop every row with a missing value?

Sometimes! But dropping rows also means losing potentially useful data.

We'll fill missing values later using information from our training data.

## 3. Exploratory Data Analysis (EDA)

Before training a model, we should understand our dataset.

We'll look at summary statistics and visualize relationships between variables.

In [ ]:
df.describe()

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df["temperature_f"], df["rides"])
plt.xlabel("Temperature (°F)")
plt.ylabel("Number of Rentals")
plt.title("Temperature vs. Bike Rentals")
plt.show()

In [ ]:
df.groupby("rain")["rides"].mean()

In [ ]:
df.groupby("rain")["rides"].mean().plot(kind="bar")

plt.ylabel("Average Rentals")
plt.title("Bike Rentals by Rain Type")
plt.show()

**Discussion:**
- Do warmer days tend to have more rentals?
- What happens when it rains?
- Does correlation always mean causation?

## 4. Feature Selection

**Features (X):** Information our model uses to make predictions.

**Target (y):** What our model is trying to predict.

For our first models:
- Features: Weather, weekend, and season
- Target: Whether the day is busy (`busy_day`)

We won't use `rides` to predict `busy_day`, since that label was created from the rental count.

In [ ]:
features = ["temperature_f", "wind_mph", "rain", "weekend", "season"]

X = df[features]
y = df["busy_day"]

X.head()

### Turning Text into Numbers

Machine learning models need numerical inputs.

We'll use **one-hot encoding** to convert categories like "Summer" and "Heavy" into numerical columns.

In [ ]:
X = pd.get_dummies(X, dummy_na=True).astype(float)
X.head()

### Training vs. Testing Data

We train models on one portion of our data and test them on observations they haven't seen.

We'll use:
- 75% for training
- 25% for testing

This helps us evaluate how well our models generalize.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

### Fill Missing Values

We'll replace missing numerical values with their averages.

Importantly, we calculate the averages using **only training data**, so we don't use information from our test set while preparing the model.

In [ ]:
train_averages = X_train.mean()

X_train = X_train.fillna(train_averages)
X_test = X_test.fillna(train_averages)

## 5. K-Nearest Neighbors (KNN)

KNN predicts by looking at similar examples.

For our model:
1. Find the 5 most similar past days.
2. Look at whether those days were busy.
3. Predict the most common answer.

We also standardize the features so variables with larger numbers don't dominate the distance calculation.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))

knn.fit(X_train, y_train)

knn_predictions = knn.predict(X_test)

print("KNN Accuracy:", accuracy_score(y_test, knn_predictions))

## 6. Decision Trees

Decision trees make predictions by learning rules from data.

For example:
- Is it raining?
- Is it a weekend?
- Is the temperature high?

We'll limit the tree's depth to keep it simple.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=3, random_state=42)

tree.fit(X_train, y_train)

tree_predictions = tree.predict(X_test)

print("Tree Accuracy:", accuracy_score(y_test, tree_predictions))

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(14, 6))
plot_tree(tree, feature_names=X_train.columns,
          class_names=tree.classes_, filled=True,
          rounded=True, fontsize=8)
plt.show()

**Discussion:**
- Which model performed better?
- Can you follow the decision tree's rules?
- Would either model always be correct?

## 7. Linear Regression

Our previous models predicted categories: Busy or Not Busy.

Now we'll predict a **number**: How many bikes will be rented?

We'll use the same features and training/testing rows, but change our target to `rides`.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

regression = LinearRegression()

regression.fit(X_train, df.loc[X_train.index, "rides"])

predictions = regression.predict(X_test)
actual = df.loc[X_test.index, "rides"]

print("Mean Absolute Error:", mean_absolute_error(actual, predictions))


### What Is Mean Absolute Error (MAE)?

MAE measures how far our predictions are from the actual values, on average.

For example, an MAE of 30 means our predictions are off by approximately 30 bike rentals on average.

**Lower is better!**

In [ ]:
plt.scatter(actual, predictions)
plt.xlabel("Actual Rentals")
plt.ylabel("Predicted Rentals")
plt.title("Actual vs. Predicted Rentals")
plt.show()

## 8. Make a Prediction!

Imagine a summer weekend with warm weather and no rain.

Will it be a busy day? How many bikes will people rent?

Let's ask our models!

In [ ]:
new_day = pd.DataFrame([{
    "temperature_f": 78,
    "wind_mph": 5,
    "rain": "Dry",
    "weekend": "Yes",
    "season": "Summer"
}])

new_day = pd.get_dummies(new_day)
new_day = new_day.reindex(columns=X_train.columns, fill_value=0).astype(float)

print("KNN:", knn.predict(new_day)[0])
print("Decision Tree:", tree.predict(new_day)[0])
print("Predicted Rentals:", round(regression.predict(new_day)[0]))

## 9. Try It Yourself!

1. Change the temperature to 40°F. What happens?
2. Change the weather to heavy rain. What happens?
3. Try changing KNN from 5 neighbors to 3.
4. Which model would you use to predict bike rentals?

### Key Takeaways

- Data science begins with asking questions.
- Cleaning and exploring data are important parts of the process.
- Different ML models solve different types of problems.
- Models should be evaluated on data they haven't seen before.